# Web Skin — 기존 B0 모델 검증 및 부분 미세조정
웹캠 **얼굴 정면** 피부 5-class 전용입니다. Hair/현미경 Skin과 섞지 않습니다.

1. 기존 데이터 ZIP을 검사합니다. 중복·라벨 충돌·평가셋 불일치가 있으면 보고서를 저장하고 중단합니다.
2. 검증된 기존 Augmented 모델의 Validation 성능을 다시 확인합니다.
3. 마지막 30개 계층 중 Batch Normalization을 제외한 계층을 10 Epoch 미세조정합니다.
4. 기존 모델과 새 모델 중 **Validation이 높은 모델**만 Test에서 평가합니다. 동률이면 기존 모델을 유지합니다.
5. 그래프, 모델, 설정, 지표, 예측 목록을 Drive의 새 폴더와 ZIP으로 저장합니다.

고정: 기존 분할, 클래스 순서, 224×224, Batch 32, CE, 기존 Augmented Train, 원본 Validation/Test.
주요 변경: 후반부 부분 미세조정. 동반 변경: Adam을 새로 구성하고 학습률 1e-5로 10 Epoch 추가.
이전 15 Epoch의 optimizer 상태를 이어받는 실험은 아닙니다.
사람·병변·세션과 증강 출처 기록이 없으면 해당 수준의 누수는 확인 불가로 기록합니다.
정상 클래스는 포함하지만 범위 밖 입력 거부 및 실제 웹캠 검증은 미구현입니다.

**Colab GPU에서 위에서 아래로 실행하세요. 오류가 나면 아래 학습 셀로 건너뛰지 마세요.**


## 1. Drive와 실행 환경
설치 후 재시작 안내가 뜨면 런타임을 재시작하고 처음부터 실행합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip -q install tensorflow==2.20.0 keras==3.13.2 scikit-learn pandas matplotlib pillow tqdm

import hashlib
import json
import platform
import shutil
import stat
import uuid
import zipfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from tqdm.auto import tqdm

if tf.__version__ != '2.20.0' or keras.__version__ != '3.13.2':
    raise RuntimeError('설치된 버전을 반영하려면 런타임을 재시작하고 처음부터 실행하세요.')
if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('Colab 런타임 유형을 GPU로 변경하세요.')
print('TensorFlow:', tf.__version__, 'Keras:', keras.__version__)


## 2. 경로 설정
기존 Drive 파일명을 자동 검색합니다. 후보가 여러 개면 임의 선택하지 않습니다. 그때 아래 두 경로에 정확한 경로를 넣습니다. 결과 ZIP은 압축 해제할 필요가 없습니다.

In [ ]:
MY_DRIVE = Path('/content/drive/MyDrive')
DATA_ZIP_OVERRIDE = ''  # 예: /content/drive/MyDrive/web_skin_processed.zip
BASELINE_OVERRIDE = ''  # web_skin_dataset_results ZIP, 폴더 또는 augmented/best_model.keras
CLASS_NAMES = ['건선', '아토피', '여드름', '정상', '주사']
CLASS_CODES = ['C0', 'C1', 'C2', 'C3', 'C4']
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
UNFREEZE_LAST_N = 30
EPOCHS = 10
LEARNING_RATE = 1e-5
EXPECTED_BASELINE_SHA256 = '3b05c59500c272600438026b758842728b21d2ad2ce376cb435f436d4167310f'
REPORTED_BASELINE = {
    'best_val_accuracy': 0.6880000233650208,
    'test_accuracy': 0.8199999928474426,
    'macro_f1': 0.8195818185502844,
}
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8]
RUN_NAME = 'partial_finetune_224_' + RUN_ID
LOCAL_ROOT = Path('/content') / ('mediflow_web_skin_' + RUN_ID)
EXTRACT_ROOT = LOCAL_ROOT / 'dataset'
RESULT_DIR = MY_DRIVE / 'mediflow_experiments' / 'web_skin' / RUN_NAME
LOCAL_ROOT.mkdir(parents=True, exist_ok=False)
EXTRACT_ROOT.mkdir()
RESULT_DIR.mkdir(parents=True, exist_ok=False)
DATA_AUDIT_OK = False
BASELINE_OK = False
SELECTION_FIXED = False
keras.utils.set_random_seed(SEED)

def save_json(name, value):
    (RESULT_DIR / name).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def unique_path(paths, label):
    candidates = sorted({p.resolve() for p in paths if p.exists()})
    if len(candidates) != 1:
        raise ValueError(f'{label} 경로를 하나로 확정할 수 없습니다. OVERRIDE에 경로를 입력하세요: {candidates}')
    return candidates[0]

def safe_extract(source, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(source) as archive:
        for item in archive.infolist():
            target = (destination / item.filename).resolve()
            if not target.is_relative_to(destination) or stat.S_ISLNK(item.external_attr >> 16):
                raise ValueError(f'허용하지 않는 ZIP 항목: {item.filename}')
        archive.extractall(destination)

def export_zip():
    path = Path(shutil.make_archive(str(LOCAL_ROOT / RUN_NAME), 'zip',
                                   root_dir=RESULT_DIR.parent, base_dir=RESULT_DIR.name))
    destination = RESULT_DIR.parent / path.name
    # 재실행해도 기존 ZIP은 덮어쓰지 않는다.
    if destination.exists():
        destination = destination.with_name(destination.stem + '_' + uuid.uuid4().hex[:8] + '.zip')
    with path.open('rb') as source, destination.open('xb') as output:
        shutil.copyfileobj(source, output)
    if sha256(path) != sha256(destination):
        raise IOError('Drive ZIP 복사 검증 실패')
    print('Drive 결과 ZIP:', destination)
    return destination

save_json('run_start.json', {
    'domain': 'web_skin', 'run_id_utc': RUN_ID,
    'created_from_commit': 'b5faa6d937229a49e9d62541a30e39f3b75a3c77',
    'notebook_state': '새 미커밋 노트북; 정확한 셀 코드를 code_snapshot.py에 기록',
    'class_names': CLASS_NAMES, 'reported_baseline': REPORTED_BASELINE,
})
print('결과 저장:', RESULT_DIR)


## 3. 데이터 ZIP 복사·압축 해제
기존 데이터는 삭제하거나 재분할하지 않습니다.

In [ ]:
if DATA_ZIP_OVERRIDE:
    DATA_ZIP_PATH = unique_path([Path(DATA_ZIP_OVERRIDE)], '데이터')
else:
    DATA_ZIP_PATH = unique_path(
        [p for p in MY_DRIVE.rglob('web_skin_processed*') if p.is_file() and zipfile.is_zipfile(p)],
        '데이터 ZIP',
    )
if not zipfile.is_zipfile(DATA_ZIP_PATH):
    raise ValueError('선택한 데이터가 ZIP이 아닙니다.')
LOCAL_DATA_ZIP = LOCAL_ROOT / 'web_skin_processed.zip'
shutil.copyfile(DATA_ZIP_PATH, LOCAL_DATA_ZIP)
DATA_ZIP_SHA256 = sha256(LOCAL_DATA_ZIP)
if DATA_ZIP_SHA256 != sha256(DATA_ZIP_PATH):
    raise IOError('데이터 ZIP 복사 중 내용이 달라졌습니다.')
safe_extract(LOCAL_DATA_ZIP, EXTRACT_ROOT)
save_json('data_source.json', {'path': str(DATA_ZIP_PATH), 'sha256': DATA_ZIP_SHA256})
print('데이터:', DATA_ZIP_PATH, '\nSHA-256:', DATA_ZIP_SHA256)


## 4. 학습 전 데이터 감사
파일 내용 SHA-256과 RGB 픽셀 해시를 함께 비교합니다. 파일명이 달라도 같은 파일/같은 픽셀이면 찾습니다.
증강 Train과 원본 평가셋 사이 중복, 클래스 간 라벨 충돌, Original/Augmented 평가셋의 **라벨·개수·내용** 일치를 확인합니다.
재압축·회전·밝기 변경된 유사 이미지, 사람/병변/세션 중복, 증강 원본 연결은 이 해시 검사로 입증할 수 없습니다.
JSON/CSV 파일 목록도 남깁니다. 출처 기록이 있으면 후속 검토가 필요합니다.

문제가 있으면 `audit_summary.json`, `image_inventory.csv`, `audit_issues.csv`를 저장하고 중단합니다.

In [ ]:
DATA_AUDIT_OK = False
SPLITS = ('train', 'val', 'test')
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp'}
roots = {}
inventory = []
issues = []

for kind in ('original', 'augmented'):
    roots[kind] = unique_path(
        [p for p in EXTRACT_ROOT.rglob(kind)
         if p.is_dir() and all((p / split).is_dir() for split in SPLITS)], kind,
    )
    for split in SPLITS:
        actual_classes = sorted(p.name for p in (roots[kind] / split).iterdir() if p.is_dir())
        if actual_classes != sorted(CLASS_NAMES):
            issues.append({'type': 'class_structure', 'detail': f'{kind}/{split}: {actual_classes}'})
        for cls in CLASS_NAMES:
            paths = sorted(p for p in (roots[kind] / split / cls).rglob('*') if p.is_file())
            images = [p for p in paths if p.suffix.lower() in EXTENSIONS]
            unsupported = [p for p in paths if p.suffix.lower() in {'.webp', '.gif', '.tif', '.tiff'}]
            for path in unsupported:
                issues.append({'type': 'unsupported_image', 'detail': str(path.relative_to(EXTRACT_ROOT))})
            if not images:
                issues.append({'type': 'empty_class', 'detail': f'{kind}/{split}/{cls}'})
            for path in tqdm(images, desc=f'{kind}/{split}/{cls}'):
                try:
                    with Image.open(path) as source:
                        rgb = source.convert('RGB')
                        pixel_hash = hashlib.sha256(
                            str(rgb.size).encode('ascii') + b':' + rgb.tobytes()).hexdigest()
                    inventory.append({
                        'kind': kind, 'split': split, 'class_name': cls,
                        'path': str(path.relative_to(EXTRACT_ROOT)),
                        'sha256': sha256(path), 'pixel_sha256': pixel_hash,
                    })
                except Exception as exc:
                    issues.append({'type': 'unreadable', 'detail': f'{path}: {exc}'})

df = pd.DataFrame(inventory, columns=['kind', 'split', 'class_name', 'path', 'sha256', 'pixel_sha256'])
df.to_csv(RESULT_DIR / 'image_inventory.csv', index=False, encoding='utf-8-sig')
for key in ('sha256', 'pixel_sha256'):
    for digest, group in df.groupby(key):
        if group['split'].nunique() > 1:
            issues.append({'type': key + '_cross_split', 'detail': digest,
                           'paths': '|'.join(group['path'])})
        if group['class_name'].nunique() > 1:
            issues.append({'type': key + '_label_conflict', 'detail': digest,
                           'paths': '|'.join(group['path'])})

def counted_content(kind, split, key):
    part = df[(df.kind == kind) & (df.split == split)]
    return Counter(zip(part['class_name'], part[key]))

for split in ('val', 'test'):
    if counted_content('original', split, 'sha256') != counted_content('augmented', split, 'sha256'):
        issues.append({'type': 'evaluation_set_mismatch', 'detail': split})
# 원본 Train이 증강 Train에 보존됐는지 확인한다.
if counted_content('original', 'train', 'sha256') - counted_content('augmented', 'train', 'sha256'):
    issues.append({'type': 'missing_original_train', 'detail': 'Augmented Train에 원본이 누락됨'})

counts = df.groupby(['kind', 'split', 'class_name']).size().rename('count').reset_index()
counts.to_csv(RESULT_DIR / 'dataset_counts.csv', index=False, encoding='utf-8-sig')
metadata_files = sorted(str(p.relative_to(EXTRACT_ROOT)) for p in EXTRACT_ROOT.rglob('*')
                        if p.is_file() and p.suffix.lower() in {'.json', '.csv'})
limitations = [
    '사람·병변·촬영 세션 식별 정보와 이미지의 대응 관계를 검증하지 못함',
    '기존 증강본의 원본 연결 미검증; 해시가 다른 파생본은 발견하지 못할 수 있음',
    '재압축·회전·색 변경된 유사 이미지 중복 및 라벨 의미의 정확성 미검증',
    '기존 학습 시점 데이터 ZIP 해시가 없어 완전한 동일 분할 재현을 입증할 수 없음',
    '실제 웹캠 검증 없음; 공개 데이터 결과로만 해석',
]
audit = {
    'data_zip_sha256': DATA_ZIP_SHA256, 'issues_count': len(issues),
    'checks_passed': not issues, 'metadata_files': metadata_files, 'limitations': limitations,
    'counts': counts.to_dict('records'),
}
pd.DataFrame(issues, columns=['type', 'detail', 'paths']).to_csv(
    RESULT_DIR / 'audit_issues.csv', index=False, encoding='utf-8-sig')
save_json('audit_summary.json', audit)
display(counts)
print('\n'.join(limitations))
if issues:
    display(pd.DataFrame(issues).head(30))
    export_zip()
    raise ValueError('데이터 감사에서 문제가 발견되어 학습 중단. 검사 ZIP을 공유하세요.')
DATA_AUDIT_OK = True
print('기계적 중복 검사 통과. 모든 종류의 누수가 배제됐다는 뜻은 아닙니다.')


## 5. 기존 Augmented 모델 찾기·검증
Drive의 `web_skin_dataset_results` ZIP/폴더 또는 기존 `web_skin_training_results` 폴더를 찾습니다. 로컬에서 확인한 모델 해시와 다르면 진행하지 않습니다.

In [ ]:
assert DATA_AUDIT_OK, '먼저 데이터 검사 셀을 통과해야 합니다.'
BASELINE_OK = False
if BASELINE_OVERRIDE:
    source = unique_path([Path(BASELINE_OVERRIDE)], '기존 모델')
else:
    names = {'web_skin_dataset_results', 'web_skin_dataset_results.zip',
             'web_skin_training_results', 'web_skin_training_results.zip'}
    source = unique_path([p for p in MY_DRIVE.rglob('*') if p.name in names], '기존 결과')
if source.is_dir():
    search_root = source
elif zipfile.is_zipfile(source) and source.suffix.lower() != '.keras':
    search_root = LOCAL_ROOT / 'baseline_results'
    search_root.mkdir()
    safe_extract(source, search_root)
else:
    search_root = None
if search_root is not None:
    baseline_path = unique_path(
        [p for p in search_root.rglob('best_model.keras') if p.parent.name.lower() == 'augmented'],
        'Augmented 최고 모델')
else:
    baseline_path = source
BASELINE_SHA256 = sha256(baseline_path)
if BASELINE_SHA256 != EXPECTED_BASELINE_SHA256:
    raise ValueError('검증한 Web Skin Augmented 모델과 해시가 다릅니다. 모델을 다시 확인하세요.')
metadata_path = baseline_path.parent / 'results.json'
if metadata_path.exists():
    metadata = json.loads(metadata_path.read_text(encoding='utf-8-sig'))
    if metadata.get('classes') != CLASS_NAMES:
        raise ValueError('기존 결과 JSON의 클래스 순서가 다릅니다.')
    save_json('source_results.json', metadata)
else:
    save_json('source_results.json', {
        'classes': CLASS_NAMES, 'reported_baseline': REPORTED_BASELINE,
        'source': '로컬 결과와 일치하는 모델 SHA-256으로 식별; Drive JSON 미제공',
    })
LOCAL_BASELINE = LOCAL_ROOT / 'baseline.keras'
shutil.copyfile(baseline_path, LOCAL_BASELINE)
model = keras.models.load_model(LOCAL_BASELINE, compile=False)
assert model.input_shape == (None, 224, 224, 3)
assert model.output_shape == (None, 5)
backbones = [layer for layer in model.layers
             if isinstance(layer, keras.Model) and 'efficientnet' in layer.name.lower()]
if len(backbones) != 1:
    raise ValueError('EfficientNet Backbone을 하나로 확정할 수 없습니다.')
backbone = backbones[0]
assert any(isinstance(layer, keras.layers.Rescaling)
           and np.allclose(layer.get_config()['scale'], 1/255)
           for layer in backbone.layers), '내부 Rescaling 계약 불일치'

def make_dataset(split, shuffle=False):
    return keras.utils.image_dataset_from_directory(
        roots['augmented'] / split, labels='inferred', label_mode='categorical',
        class_names=CLASS_NAMES, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
        shuffle=shuffle, seed=SEED if shuffle else None,
    )
train_ds = make_dataset('train', True).prefetch(tf.data.AUTOTUNE)
val_ds = make_dataset('val').prefetch(tf.data.AUTOTUNE)
model.compile(optimizer=keras.optimizers.Adam(1e-4),
              loss='categorical_crossentropy', metrics=['accuracy'])
baseline_val = model.evaluate(val_ds, return_dict=True, verbose=1)
save_json('baseline_validation.json', {
    'newly_measured': baseline_val, 'reported': REPORTED_BASELINE['best_val_accuracy'],
    'model_sha256': BASELINE_SHA256,
})
if abs(baseline_val['accuracy'] - REPORTED_BASELINE['best_val_accuracy']) > 1e-5:
    export_zip()
    raise ValueError('기존 Validation 점수가 재현되지 않았습니다. 보고서 확인 전 학습하지 않습니다.')
BASELINE_OK = True
print('기존 모델 Validation 재현:', baseline_val)


## 6. 부분 미세조정 10 Epoch
원래 모델은 보존합니다. CSV 로그와 최고 체크포인트를 매 Epoch Drive에 저장합니다.

In [ ]:
assert DATA_AUDIT_OK and BASELINE_OK
SELECTION_FIXED = False
backbone.trainable = True
for index, layer in enumerate(backbone.layers):
    layer.trainable = (index >= len(backbone.layers) - UNFREEZE_LAST_N
                       and not isinstance(layer, keras.layers.BatchNormalization))
model.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE),
              loss='categorical_crossentropy', metrics=['accuracy'])
config = {
    'domain': 'web_skin', 'hypothesis': '후반부 부분 미세조정이 기존 동결 모델보다 Validation을 개선한다',
    'fixed_conditions': ['existing_split', 'augmented_train', 'original_val_test',
                         'B0', '224', 'batch32', 'seed42', 'cross_entropy', 'dropout0.3'],
    'changed_variable': '마지막 30개 Layer 중 BN 제외 계층 미세조정',
    'accompanying_changes': '새 Adam, 학습률 1e-5, 추가 10 Epoch; 단순 학습시간 효과와 완전히 분리되지 않음',
    'created_from_commit': 'b5faa6d937229a49e9d62541a30e39f3b75a3c77',
    'notebook_state': 'uncommitted; code_snapshot.py 참조',
    'data_zip_path': str(DATA_ZIP_PATH), 'data_zip_sha256': DATA_ZIP_SHA256,
    'baseline_model_source': str(source), 'baseline_model_sha256': BASELINE_SHA256,
    'class_names': CLASS_NAMES, 'normal_class_included': True,
    'out_of_scope_handling': '미구현', 'image_size': list(IMAGE_SIZE),
    'input_dtype': 'float32', 'input_pixel_range': [0, 255], 'external_normalization': False,
    'seed': SEED, 'batch_size': BATCH_SIZE, 'epochs': EPOCHS, 'learning_rate': LEARNING_RATE,
    'unfreeze_last_n': UNFREEZE_LAST_N, 'batch_normalization_frozen': True,
    'trainable_backbone_layers': [layer.name for layer in backbone.layers if layer.trainable],
    'selection_metric': 'validation_accuracy', 'baseline_validation': baseline_val,
    'reported_baseline': REPORTED_BASELINE, 'leakage_audit': audit,
    'environment': {'python': platform.python_version(), 'tensorflow': tf.__version__,
                    'keras': keras.__version__, 'numpy': np.__version__,
                    'devices': [str(d) for d in tf.config.list_physical_devices()]},
}
save_json('training_config.json', config)
# 실행 셀을 저장하여 Colab 설정 변경도 기록합니다.
ipython = get_ipython()
snapshot = '\n\n# ---- cell ----\n\n'.join(ipython.history_manager.input_hist_raw)
(RESULT_DIR / 'code_snapshot.py').write_text(snapshot, encoding='utf-8')
checkpoint = RESULT_DIR / 'finetuned_best.keras'
history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=[
        keras.callbacks.ModelCheckpoint(str(checkpoint), monitor='val_accuracy',
                                        mode='max', save_best_only=True),
        keras.callbacks.CSVLogger(str(RESULT_DIR / 'training_log.csv')),
    ],
)
save_json('training_history.json', {
    key: [float(v) for v in values] for key, values in history.history.items()})
fine_val = float(max(history.history['val_accuracy']))
best_epoch = int(np.argmax(history.history['val_accuracy'])) + 1
use_finetuned = fine_val > baseline_val['accuracy']
selected_source = checkpoint if use_finetuned else LOCAL_BASELINE
selected_path = RESULT_DIR / 'selected_model.keras'
if selected_path.exists():
    raise FileExistsError('이미 선택 모델이 있습니다. 새 실행으로 시작하세요.')
shutil.copyfile(selected_source, selected_path)
selection = {
    'selected': 'partial_finetuning' if use_finetuned else 'baseline',
    'baseline_validation': float(baseline_val['accuracy']),
    'finetuned_validation': fine_val, 'finetuned_best_epoch': best_epoch,
    'selected_val_accuracy': fine_val if use_finetuned else float(baseline_val['accuracy']),
    'selected_model_sha256': sha256(selected_path),
    'tie_policy': 'baseline 유지',
}
save_json('selection.json', selection)
SELECTION_FIXED = True
print(selection)


## 7. 선택된 모델 Test 평가·시각화
최고 모델 선택 이후 한 번만 Test 예측을 실행합니다. 전체 학습곡선, Validation 비교, 혼동행렬, 클래스별 F1을 저장합니다. 기존 Test는 재측정 결과와 구분합니다.

In [ ]:
assert DATA_AUDIT_OK and BASELINE_OK and SELECTION_FIXED
if (RESULT_DIR / 'evaluation.json').exists():
    raise RuntimeError('이미 평가가 완료됐습니다. Test 재평가 대신 결과 파일을 확인하세요.')
chosen = keras.models.load_model(selected_path, compile=False)
test_raw = make_dataset('test')
test_paths = test_raw.file_paths
y_true, probabilities = [], []
for images, labels in test_raw.prefetch(tf.data.AUTOTUNE):
    probabilities.extend(chosen(images, training=False).numpy())
    y_true.extend(np.argmax(labels.numpy(), axis=1))
probs = np.asarray(probabilities)
truth = np.asarray(y_true)
pred = np.argmax(probs, axis=1)
evaluation = {
    'test_accuracy': float(accuracy_score(truth, pred)),
    'macro_f1': float(f1_score(truth, pred, labels=list(range(5)), average='macro', zero_division=0)),
    'test_count': len(truth), 'selection': selection, 'reported_baseline': REPORTED_BASELINE,
    'interpretation': '기존 지표는 저장 보고값; 이번 Test는 선택된 모델만 측정. 원 학습 ZIP 동일성 미입증',
}
save_json('evaluation.json', evaluation)
report = classification_report(truth, pred, labels=list(range(5)), target_names=CLASS_NAMES,
                               output_dict=True, zero_division=0)
pd.DataFrame(report).T.to_csv(RESULT_DIR / 'classification_report.csv', encoding='utf-8-sig')
pd.DataFrame({
    'path': [str(Path(p).relative_to(EXTRACT_ROOT)) for p in test_paths],
    'true_index': truth, 'pred_index': pred,
    **{f'prob_C{i}': probs[:, i] for i in range(5)},
}).to_csv(RESULT_DIR / 'test_predictions.csv', index=False, encoding='utf-8-sig')
cm = confusion_matrix(truth, pred, labels=list(range(5)))
pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
    RESULT_DIR / 'confusion_matrix.csv', encoding='utf-8-sig')
save_json('class_names.json', CLASS_NAMES)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
epochs = np.arange(1, EPOCHS + 1)
for ax, metric in zip(axes, ('accuracy', 'loss')):
    ax.plot(epochs, history.history[metric], label='Train')
    ax.plot(epochs, history.history['val_' + metric], label='Validation')
    ax.axhline(baseline_val[metric], ls='--', color='gray', label='Baseline Validation')
    ax.set(xlabel='Additional Epoch', ylabel=metric, title='Web Skin Partial Fine-tuning: ' + metric)
    ax.grid(alpha=0.3)
    ax.legend()
fig.tight_layout()
fig.savefig(RESULT_DIR / 'training_curves.png', dpi=200)
plt.show()
plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].bar(['Baseline', 'Fine-tuned'], [baseline_val['accuracy'], fine_val])
axes[0].set(ylim=(0, 1), title='Validation selection', ylabel='Accuracy')
for i, value in enumerate([baseline_val['accuracy'], fine_val]):
    axes[0].text(i, value + 0.01, str(value), ha='center', fontsize=8)
axes[1].imshow(cm, cmap='Blues')
axes[1].set(xticks=range(5), yticks=range(5), xticklabels=CLASS_CODES,
            yticklabels=CLASS_CODES, xlabel='Predicted', ylabel='True', title='Selected model Test')
for i in range(5):
    for j in range(5):
        axes[1].text(j, i, str(cm[i, j]), ha='center', va='center',
                     color='white' if cm[i, j] > cm.max()/2 else 'black')
f1_values = [report[name]['f1-score'] for name in CLASS_NAMES]
axes[2].bar(CLASS_CODES, f1_values)
axes[2].set(ylim=(0, 1), title='Selected model class F1')
fig.tight_layout()
fig.savefig(RESULT_DIR / 'performance_dashboard.png', dpi=200)
plt.show()
plt.close(fig)
print(dict(zip(CLASS_CODES, CLASS_NAMES)))
print(evaluation)


## 8. Drive ZIP 백업
아래에 표시되는 ZIP을 내려받아 공유하면 다음 실험을 정합니다. 학습을 재실행할 필요가 없습니다.

In [ ]:
assert SELECTION_FIXED
(RESULT_DIR / 'code_snapshot.py').write_text(
    '\n\n# ---- cell ----\n\n'.join(get_ipython().history_manager.input_hist_raw),
    encoding='utf-8')
save_json('artifact_manifest.json', {
    p.name: {'sha256': sha256(p), 'bytes': p.stat().st_size}
    for p in sorted(RESULT_DIR.iterdir()) if p.is_file() and p.name != 'artifact_manifest.json'
})
RESULT_ZIP = export_zip()
print('완료. 폴더:', RESULT_DIR)
print('공유할 파일:', RESULT_ZIP)
